In [1]:
import numpy as np
from scipy.stats import skellam
from scipy.optimize import brentq

def H_star_meanfield(alpha, H_min=0.5, H_max=1.0, eps=1e-10):
    """
    Mean-field attractor H* for the closure
        H = P(F > 0) + H P(F = 0),
    where F = U - D, U ~ Pois(alpha), D ~ Pois(alpha H).

    Parameters
    ----------
    alpha : float
        Clause density parameter.
    H_min, H_max : float
        Search interval for the majority-1 branch.
    eps : float
        Small offset to avoid endpoint issues.

    Returns
    -------
    Hstar : float
        Fixed-point homogeneity on the branch H >= 1/2.
    """

    if alpha <= 0:
        raise ValueError("alpha must be positive")

    a = max(H_min, 0.5) + eps
    b = min(H_max, 1.0) - eps
    if a >= b:
        raise ValueError("Need H_min < H_max within [0.5, 1.0]")

    def fixed_point_residual(H):
        mu1 = alpha
        mu2 = alpha * H
        p_eq = skellam.pmf(0, mu1, mu2)   # P(F = 0)
        p_pos = skellam.sf(0, mu1, mu2)   # P(F > 0)
        G = p_pos + H * p_eq
        return G - H

    fa = fixed_point_residual(a)
    fb = fixed_point_residual(b)

    if fa == 0:
        return a
    if fb == 0:
        return b

    if fa * fb > 0:
        grid = np.linspace(a, b, 400)
        vals = np.array([fixed_point_residual(x) for x in grid])
        idx = np.where(vals[:-1] * vals[1:] <= 0)[0]
        if len(idx) == 0:
            raise RuntimeError("No fixed point found on [0.5, 1]. Try a wider model/closure.")
        i = idx[0]
        return brentq(fixed_point_residual, grid[i], grid[i+1])

    return brentq(fixed_point_residual, a, b)


def H_star_grid(alphas, **kwargs):
    """
    Evaluate H* for a list/array of alpha values.
    Returns a dict {alpha: H*}.
    """
    alphas = np.atleast_1d(alphas).astype(float)
    return {float(a): H_star_meanfield(float(a), **kwargs) for a in alphas}

In [9]:
H_star_meanfield(200)

0.8852822982120976

In [11]:
H_star_grid([0.5, 1.0, 2.0, 3.0, 4.0,10.0,20,100])

{0.5: 0.6342744277759711,
 1.0: 0.6491117391958152,
 2.0: 0.6724037042637862,
 3.0: 0.6892131646839185,
 4.0: 0.7021245509794815,
 10.0: 0.7466765201066299,
 20.0: 0.7817792315712321,
 100.0: 0.8577592160171177}